# GPTChallenge: diagnóstico a partir de HCE

Vamos a trabajar con el corpus CodEsp (textos de historial clínico etiquetados con sus códigos CIE-10 Diagnóstico)

In [45]:
import pandas as pd
import os, re
import numpy as np

pd.options.display.max_colwidth = None

In [46]:
#los códigos están en un TSV con un código por línea
path_train = "data/train/train.tsv"
train_diag = pd.read_csv(path_train, sep="\t", header=None, names=["archivo", "codigo"])
train_diag.info()

<class 'pandas.DataFrame'>
RangeIndex: 8316 entries, 0 to 8315
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   archivo  8316 non-null   str  
 1   codigo   8316 non-null   str  
dtypes: str(2)
memory usage: 130.1 KB


In [47]:
train_diag['codigo'].value_counts()

codigo
r52        163
r69        150
r50.9      142
i10        116
r59.9       95
          ... 
s92.322      1
d25.9        1
e72.09       1
d13.5        1
n81.2        1
Name: count, Length: 2194, dtype: int64

In [48]:
#cogemos la categoría superior de cada código y las agrupamos
train_diag['cat'] = train_diag['codigo'].str.extract(r'(\w\d\d)')
print(train_diag['cat'].value_counts())
train_diag['cat'].nunique()

cat
r52    163
r10    163
r59    160
r69    150
r50    144
      ... 
v19      1
t21      1
w40      1
d25      1
n81      1
Name: count, Length: 918, dtype: int64


918

In [49]:
categories=train_diag['cat'].value_counts()[:10]
top_categorias = categories.index.to_list()
print(top_categorias)

['r52', 'r10', 'r59', 'r69', 'r50', 'r60', 'i10', 'r11', 'd49', 'n28']


In [50]:
#seleccionamos sólo las etiquetas de este subconjunto
train_diag = train_diag[np.isin(train_diag['cat'], top_categorias)]

In [51]:
#cargamos los dos conjuntos de train
path = 'data/train/text_files/'

corpus = []
for f in [f for f in os.listdir(path) if f.endswith('.txt')]:
    with open(os.path.join(path, f), encoding="utf8") as text:
        texto = text.read()
    #buscamos códigos
    file = f[:-4]
    codigos = train_diag.query('archivo==@file')['cat'].to_list()
    codigos = list(set(codigos))
    if codigos:
        corpus.append({
            'archivo': file,
            'texto': texto,
            'codigos': codigos
        })
    
df_train = pd.DataFrame(corpus).set_index('archivo')
df_train.info()

<class 'pandas.DataFrame'>
Index: 562 entries, S0004-06142005000700014-1 to S2340-98942015000100005-1
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   texto    562 non-null    str   
 1   codigos  562 non-null    object
dtypes: object(1), str(1)
memory usage: 13.2+ KB


In [52]:
df_train.sample(3)

,texto,codigos
archivo,,
S1137-66272012000100011-1,"Varón de 34 años, fumador de 15 cigarros al día, con antecedentes de esofagitis péptica, hernia de hiato y asma bronquial no alérgico. Padre diabético. Cuadro de dolor abdominal intenso en flanco izdo, heces pastosas y aumento de los ruidos abdominales, que es valorado en atención primaria interpretando inicialmente el cuadro como posible gastroenteritis aguda. La exploración evidenciaba leve dolor a la palpación en hemiabdomen izdo y discreto aumento del peristaltismo. Ante la persistencia del dolor, se solicitan diversas pruebas complementarias.\nMientras está pendiente de nueva valoración, 10 días después de su visita a atención primaria, acude al servicio de Urgencias hospitalarias por presentar de nuevo episodio de intenso dolor abdominal en flanco izquierdo, acompañado en esta ocasión de diaforesis, nerviosismo y palidez cutánea. Valorado en tres ocasiones en servicios de urgencias, es diagnosticado y tratado como cólico renal. Presión arterial de 136/95 mm de Hg; frecuencia cardiaca: 89 l/min; temperatura: 37,7o; sudoroso. A la exploración abdominal presenta dolor en flanco izquierdo con sucusión renal izquierda positiva. El resto de la exploración física es normal. Las radiografías de tórax y abdomen son normales. Se objetiva mínima leucocitosis (10.500), leve hiponatremia y PCR=13.\nAl no mostrar mejoría clínica con tratamiento sintomático se realiza ecografía abdomino-pélvica donde se evidencia una tumoración en glándula suprarrenal izquierda. Tras este resultado se pide TAC donde se objetiva una masa de 6,5 cm de diámetro dependiente de la glándula suprarrenal izquierda que plantea realizar diagnóstico diferencial entre feocromocitoma o carcinoma suprarrenal.\n\nTras este resultado se deriva al servicio de Urología y de éste al de Medicina Interna para estudio funcional, en el intento de descartar feocromocitoma. La determinación de metanefrinas y catecolaminas en orina muestra una lectura de más de cinco veces el valor normal para metanefrinas totales en orina y para catecolaminas libres urinarias. Se diagnostica de masa suprarrenal izquierda funcionante, con marcada elevación de metanefrinas por probable feocromocitoma. Una vez integrado el diagnóstico de feocromocitoma se decide practicar adrenalectomía total izquierda laparoscópica previo bloqueo con dosazoxina 4 mg/día. El análisis anatomopatológico de los hallazgos quirúrgicos demuestra que se trata de un feocromocitoma maligno de 130 g y 6 x 6 cm, con extensa necrosis, marcado pleomorfismo celular, abundantes mitosis atípicas (<50 x 10 cga), invasión capsular y un trombo tumoral en una vena de grueso calibre (la vena clampada), sin extensión extraadrenal. El periodo transoperatorio transcurre sin complicaciones. Durante todo el proceso el paciente presenta cifras tensionales normales con buen estado general, y no se lleva a cabo ningún tratamiento.\n\n","[r10, r52]"
S0004-06142007000700014-1,"Paciente de 64 años, alérgico a penicilina y con recambio valvular aórtico por endocarditis que consultó por aparición de masa peneana de crecimiento progresivo en las últimas semanas. A la exploración física destacaba una formación excrecente y abigarrada en glande, que deformaba meato, con áreas ulceradas cubiertas de fibrina. Se palpaban adenopatías fijas y duras en ambas regiones inguinales. La radiografía de tórax y el TAC abdomino-pélvico confirmaron la presencia de adenopatías pulmonares e inguinales de gran tamaño. Con el diagnóstico de neoplasia de pene, se practicó penectomía parcial con margen de seguridad. La anatomía patológica demostró que se trataba de un sarcoma pleomórfico de pene con diferenciación osteosarcomatosa y márgenes libres de afectación. Se decidió tratamiento con dos líneas de quimioterapia consistente en adriamicina e ifosfamida pero no hubo respuesta. Ingresó de nuevo con recidiva local sangrante de gran tamaño y crecimiento rápido que provocaba obstrucción de meato con insuficiencia renal

## Cargar los textos del conjunto de test

In [53]:
path_test = "data/test/test.tsv"
test_diag = pd.read_csv("data/test/test.tsv", sep="\t", header=None, names=["archivo"])
test_diag.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   archivo  192 non-null    str  
dtypes: str(1)
memory usage: 1.6 KB


In [54]:
import pandas as pd

# Ejemplo: df['filename'] contiene nombres como "archivo1.txt", "archivo2.txt", etc.

for i, elem in enumerate(test_diag["archivo"]):
    with open(f"./data/test/text_files/{elem}.txt", "r", encoding="utf-8") as f:
        contenido = f.read()
    test_diag.at[i, "texto"] = contenido


## Binarizar las etiquetas

In [55]:
# para entrenar un clasificador multi-etiqueta generamos una matriz binaria de las etiquetas
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(df_train['codigos'])

#Guardamos las clases utilizadas en el conjunto de train
clases = mlb.classes_
num_classes = clases.shape
print(num_classes[0])

10


## Procesamiento del lenguaje natural

## Modelos

## Guardar predicciones de Test